# Summary of chapter 1

## Language models ==> Large Language models ==> Fundation models

A Language model encordes statistical information about one or more languages, it is able to predict what word is most likely to come after a sequence of words or before.

### Masked Language Model
Masked language model (like BERT): predicts missing tokens anywhere in a sequence using the context on both sides, before and after the gap. It's good for understanding tasks such as classification, sentiment and code debugging. e.g "My favorite ___ is blue" 

### Auto regressive language model
Autoregressive language model (like GPT or Claude): predicts only the next token, using the tokens before it. This is the type that generates text, so it's the type most foundation models today use. e.g My favorite color is ____

### Self-supervision: 
what makes the step up to LLMs possible. The labels come from the text itself, so models can train on huge datasets without anyone labeling them by hand.

A token is the basic unit of a Language model, it is approximately 75% of a word. Tokenization is the basis of breaking text into tokens.

### Advantages of tokens:
1. Tokens allow models to break words into meaningful components
2. There are fewer tokens than words so this makes the model vocabulary size smaller
3. Tokens allow the model to process unknown words 

### Example: BERT as a masked language model
BERT sees the words on **both sides** of `[MASK]` and guesses what fits in the gap.

In [3]:
# Install the two libraries we need (only run this once):
#   transformers -> Hugging Face's library for downloading and using AI models
#   torch        -> PyTorch, the math engine that actually runs the models
# The % sign means "install into the Python this notebook is using".
# -q means "quiet": show less text while installing.
%pip install -q transformers torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Update pip (Python's package installer) to its newest version.
# Optional: only needed if the install above complained about an old pip.
# Tip: use %pip (with the %) inside notebooks so it updates the right Python.
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [4]:
# "import" brings in code someone else wrote so we can use it.
# pipeline is a shortcut that bundles a model + everything needed to use it.
from transformers import pipeline

# Create a "fill in the blank" tool using BERT.
#   "fill-mask"          -> the task: guess the word hidden by [MASK]
#   "bert-base-uncased"  -> which model to use ("uncased" = ignores capital letters)
# The first time, this downloads the model (~440 MB). After that it loads from disk.
unmasker = pipeline("fill-mask", model="bert-base-uncased")

# Our test sentence. [MASK] marks the word we want BERT to guess.
sentence = "The doctor wrote a [MASK] for the patient."

# Ask BERT for its top 5 guesses. It gives back a list, one item per guess.
# The "for" loop goes through that list one guess at a time.
for guess in unmasker(sentence, top_k=5):
    # Each guess is a dictionary (a set of labelled values). We use two of them:
    #   guess['token_str'] -> the word BERT guessed
    #   guess['score']     -> how confident BERT is (0 to 1; 1 = 100% sure)
    # The f"..." string lets us put values inside text using {curly braces}:
    #   :<15  -> pad the word to 15 characters so the columns line up
    #   :.3f  -> show the score with 3 decimal places
    print(f"{guess['token_str']:<15} {guess['score']:.3f}")

/Users/robertkigobe/Documents/My_Research/jira_rag_assistant/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1026 19:18:07.112000 70323 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 17598.97it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


letter          0.158
report          0.142
note            0.106
prescription    0.070
statement       0.033


**Why context on both sides matters:** change only the words *after* the mask and BERT's guesses change too.
An autoregressive model (GPT) reading left to right would only see `"I went to the"` and could not use the rest.

In [7]:
# Two sentences that are identical BEFORE the [MASK] but different AFTER it.
# If BERT gives different guesses, it proves it reads the words after the gap too.
# The square brackets [ ] make a list; the loop takes one sentence at a time.
for sentence in [
    "I went to the [MASK] to eat some bread.",
    "I went to the [MASK] to watch a movie.",
]:
    # Reuse the unmasker from the cell above (no need to load BERT again).
    # This line is a "list comprehension": a short way to build a list.
    # In plain words: "for each guess g, keep only its word (token_str)".
    top = [g["token_str"] for g in unmasker(sentence, top_k=3)]

    # Print the sentence (padded to 45 characters) followed by the 3 guesses.
    print(f"{sentence:<45} -> {top}")

I went to the [MASK] to eat some bread.       -> ['kitchen', 'market', 'bathroom']
I went to the [MASK] to watch a movie.        -> ['bathroom', 'theater', 'cinema']


### Example of Autoregressive language model


In [8]:
from transformers import pipeline

# Create a text generator using GPT-2, a small autoregressive model (~500 MB).
# "text-generation" -> the task: continue the text we give it.
generator = pipeline("text-generation", model="gpt2")

# Give GPT-2 the start of a sentence and let it continue.
#   max_new_tokens=10 -> add at most 10 new tokens (roughly 7-8 words)
#   do_sample=False   -> always pick the single most likely next token,
#                        so you get the same answer every time you run it
out = generator("I went to the", max_new_tokens=10, do_sample=False)

# The result is a list holding one dictionary.
#   out[0]                    -> the first (and only) result in the list
#   ["generated_text"]        -> the full text: your prompt + what GPT-2 added
print(out[0]["generated_text"])

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 5693.40it/s]


I went to the hospital and was told that I was going to be


In [10]:
# This cell does the same thing as the one above, but step by step,
# so you can watch GPT-2 add ONE token at a time (that's "autoregressive").

# AutoTokenizer        -> turns text into token numbers (and back again)
# AutoModelForCausalLM -> loads a model that predicts the next token
#                         ("causal LM" is another name for autoregressive)
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch  # PyTorch: we use it to join lists of numbers (tensors) together

# Load GPT-2's tokenizer and the model itself.
tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

# Turn our starting text into token numbers.
#   return_tensors="pt" -> give the numbers back in PyTorch format
#   .input_ids          -> the token numbers themselves
# Try: print(ids) to see that "My name" becomes a couple of numbers.
ids = tok("My name", return_tensors="pt").input_ids

# Repeat 6 times, adding one new token each time.
# The _ means "I don't need the loop counter, just repeat".
for _ in range(6):
    # 1. Ask the model to score EVERY word in its vocabulary (~50,000)
    #    as a possible next token. "logits" are those raw scores.
    #    [0, -1] means: first sentence, last position -> the "what comes next" scores.
    logits = model(ids).logits[0, -1]

    # 2. Pick the token with the highest score (argmax = "position of the biggest").
    #    .unsqueeze(0) just reshapes the number so it can be joined on below.
    next_id = logits.argmax().unsqueeze(0)

    # 3. Stick the new token onto the end of our sequence.
    #    Next time round the loop, the model sees this longer sequence.
    ids = torch.cat([ids, next_id.unsqueeze(0)], dim=1)

    # 4. Turn the numbers back into text and print it,
    #    so you see the sentence grow by one token each line.
    print(tok.decode(ids[0]))

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 15576.95it/s]


My name is
My name is John
My name is John.
My name is John. I
My name is John. I'm
My name is John. I'm a


### Supervision vs self-supervision
**Supervised:** humans label each example by hand (expensive, slow).

**Self-supervised:** the labels come from the data itself, e.g. the next word in a sentence (free, scales to the whole internet). This is how GPT-style models are trained.

In [11]:
# SUPERVISED learning: a HUMAN had to write the label for every example.
# Here someone read each review and labelled it "positive" or "negative".
# Each item is a pair in round brackets ( ): (input, label)
labelled_reviews = [
    ("I loved this film",        "positive"),
    ("Terrible acting, so boring", "negative"),
    ("Best meal I've ever had",  "positive"),
    ("The service was awful",    "negative"),
]

# Show each example: the model would learn  input -> label
# "for text, label in ..." unpacks each pair into two variables.
for text, label in labelled_reviews:
    print(f"{text:<28} -> {label}")

# The problem: labelling millions of examples by hand is slow and expensive.

I loved this film            -> positive
Terrible acting, so boring   -> negative
Best meal I've ever had      -> positive
The service was awful        -> negative


In [12]:
# SELF-SUPERVISED learning: NO human labels needed.
# We take ordinary text and let the text itself provide the answers.
sentence = "I love street food in Kampala"

# Split the sentence into words (real models use tokens, but words are easier to read).
words = sentence.split()

# For each position, the input is "the words so far" and the label is "the next word".
# range(1, len(words)) counts 1, 2, 3, ... up to the number of words minus 1.
for i in range(1, len(words)):
    context = " ".join(words[:i])   # words[:i] -> the first i words, joined back into text
    next_word = words[i]            # the word that comes right after -> this is the label
    print(f"{context:<25} -> {next_word}")

# One sentence just gave us 5 training examples for free!
# Do this over the whole internet and you get trillions of examples,
# which is what makes training LARGE language models possible.

I                         -> love
I love                    -> street
I love street             -> food
I love street food        -> in
I love street food in     -> Kampala


## Large Language models to Foundation models

## From language models to LARGE language models

A language model becomes a **large** language model when two things grow together:

1. **Parameters**: the numbers the model learns during training. More parameters give it more capacity to learn patterns in language.
2. **Training data**: a bigger model needs more data to learn from, otherwise its extra capacity is wasted.

**Self-supervision is the bridge.** With supervised learning, the amount of data is limited by how fast humans can label it. With self-supervision, any text works as training data (books, websites, code), so data can grow to trillions of tokens. That much data is what makes very large models worth building.

### How "large" has grown
| Model | Year | Parameters |
|---|---|---|
| GPT-1 | 2018 | 117 million |
| BERT-base (my masked example) | 2018 | 110 million |
| GPT-2 (my autoregressive example uses the small 124M version) | 2019 | 1.5 billion |
| GPT-3 | 2020 | 175 billion |

"Large" is relative: what counts as large today will look small in a few years.

### Why size matters in practice
- **Memory ≈ parameters × bytes per parameter.** BERT-base at 4 bytes per parameter is about 440 MB; a 7B model is about 28 GB (quantization shrinks this).
- Larger models are better at following instructions and handling new tasks, but they cost more to train and run.

**Summary:** Language model + self-supervision → lots of data → more parameters → Large Language Model

## From large language models to FOUNDATION models

LLMs only work with text. But people learn from more than text (we see, hear and read), so AI models expanded to handle other kinds of data too.

### 1. More than text: multimodal models
A **modality** is a type of data: text, images, audio, video.
- A model that works with more than one modality is **multimodal**.
- A large model that does this is a **Large Multimodal Model (LMM)**. Examples: GPT-4V, Gemini, and Claude (which can read images).

**Self-supervision still applies.** For example, OpenAI's CLIP (2021) learned from about 400 million (image, caption) pairs found on the internet. Nobody labelled them by hand; the caption written next to an image serves as its label.

### 2. Why "foundation"?
These models are called **foundation models** because:
- they are the **base** that many different applications are built on, and
- one general model can handle many tasks that used to need separate, specially trained models (translation, sentiment, summarising, describing images...).

Before, there was one model per task. Now there is one foundation model, adapted to many tasks.

### 3. Adapting a foundation model to your task
You rarely train a foundation model yourself. Instead you adapt one that already exists:
- **Prompt engineering**: give clear instructions and examples in the input.
- **RAG (retrieval-augmented generation)**: give the model relevant documents to use when answering.
- **Finetuning**: train it a little more on your own data.

Building applications this way is what the book calls **AI engineering**.

**Summary:** Language model → (self-supervision + scale) → Large Language Model → (more modalities + general purpose) → Foundation Model → (adapted by us) → AI applications

## From foundation models to AI ENGINEERING

**AI engineering** is the process of building applications on top of foundation models that already exist.
Instead of training a model from scratch, you take a ready-made model (often through an API, "model as a service") and adapt it to your problem.

### Why AI engineering took off
1. **General-purpose capabilities**: one foundation model can do many tasks, so more applications become possible.
2. **More investment**: companies are spending heavily on AI, so models keep getting better and cheaper to use.
3. **Low barrier to entry**: you can build something useful with a few lines of code and an API key. No need for your own data, GPUs or ML expertise to get started.

### AI engineering vs traditional ML engineering
| | ML engineering | AI engineering |
|---|---|---|
| Starting point | Train your own model | Use an existing foundation model |
| Main work | Collecting data, training models | **Adapting** models (prompts, RAG, finetuning) |
| Model size | Usually small | Large, so more compute and slower responses (latency) |
| Outputs | Closed-ended (e.g. "positive"/"negative") | Open-ended (any text), so **evaluation is harder** |

### The AI engineering stack (three layers)
1. **Application development**: prompts, context, user interface, evaluation. Most AI engineers work here.
2. **Model development**: training, finetuning, datasets, inference optimisation (e.g. quantization).
3. **Infrastructure**: serving models, managing compute (GPUs), monitoring.

**Full chapter summary:**
Language model → (self-supervision + scale) → Large Language Model → (more modalities + general purpose) → Foundation Model → (adapted to real problems) → **AI Engineering**

## Foundation model use cases

The book groups use cases into **8 categories**, each with consumer and enterprise examples.

| Category | Consumer examples | Enterprise examples |
|---|---|---|
| **1. Coding** | Writing code with an AI assistant | Code review, generating tests, documentation |
| **2. Image & video production** | Photo/video editing, design | Ads, marketing content, product images |
| **3. Writing** | Emails, social media posts, essays | Copywriting, reports, SEO content |
| **4. Education** | Tutoring, homework help, language learning | Employee training and onboarding |
| **5. Conversational bots** | General chatbots, AI companions | Customer support, product assistants |
| **6. Information aggregation** | Summarising articles, "talk to your docs" | Summarising meetings, market research |
| **7. Data organization** | Searching photos, organising files | Extracting data from documents (invoices, forms), knowledge management |
| **8. Workflow automation** | Booking travel, planning, everyday tasks | Processing leads, data entry, **AI agents** that use tools |

### Key points
- **Coding** is one of the most popular and proven use cases so far.
- **Enterprises usually start with internal-facing, lower-risk uses** (e.g. a tool for employees to search internal documents) before customer-facing ones, because mistakes inside the company are cheaper than mistakes in front of customers.
- **Workflow automation with agents** is the most ambitious category: the model doesn't just answer, it **takes actions** (calls tools, fills forms, sends emails).
- Many use cases are built with the same techniques covered later: **prompt engineering, RAG, finetuning and agents**.

### Before building, ask
1. **Why build it?** Does it solve a real problem, or is it just to keep up?
2. **What role does AI play?** Is it critical to the product or a nice extra? Does it act on its own (proactive) or only respond (reactive)?
3. **Is a human in the loop?** Does AI make the final decision, or does it help a person decide?